<a href="https://colab.research.google.com/github/Maxxx-VS/IMA_SibADI/blob/main/ML_2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задание 2.2. Обучение нейросети с 2 входами (линейный нейрон)

In [1]:
!pip install -q onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 6.8 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/multiregress-092022.csv')
df = df.drop('i', axis=1)
x = df[['Me', 'ne']].to_numpy()
y = df.tc.to_numpy()
n = y.size

In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=True, with_std=True)
x_s = scaler.fit_transform(x)
y = y.reshape((-1, 1))
print('Средние:', scaler.mean_, 'Масштабирование:', scaler.scale_)

Средние: [ 143.18181818 1618.18181818] Масштабирование: [ 80.9683644  262.22191094]


In [5]:
np.random.seed(42)
b = np.random.rand(1)
w = np.random.randn(2, 1)

In [6]:
n_epochs = 500
losses = []

lr = 0.01
for epoch in range(n_epochs):
    yhat = b + x_s @ w
    error = yhat - y
    loss = np.mean(error**2)
    losses.append(loss)
    b_grad = 2 * error.mean()
    w_grad = 2 * (x_s.T @ error) / n
    if epoch % 50 == 0:
        print('Epoch: ', epoch, ' b_grad: ', b_grad, ' w_grad: ', w_grad, ' b: ', b, ' w: ', w)
    b -= lr * b_grad
    w -= lr * w_grad
print('Обучение закончено: ', ' b: ', b, ' w: ', w)

Epoch:  0  b_grad:  -163.03273794412345  w_grad:  [[-5.22561041]
 [ 1.0221967 ]]  b:  [0.37454012]  w:  [[-1.11188012]
 [ 0.31890218]]
Epoch:  50  b_grad:  -59.371580020838216  w_grad:  [[-1.86851743]
 [-0.11480994]]  b:  [52.20511908]  w:  [[0.52820932]
 [0.16778953]]
Epoch:  100  b_grad:  -21.62132930245531  w_grad:  [[-0.71402947]
 [-0.22177836]]  b:  [71.08024444]  w:  [[1.13087447]
 [0.26767915]]
Epoch:  150  b_grad:  -7.873832575133407  w_grad:  [[-0.28985991]
 [-0.15168913]]  b:  [77.9539928]  w:  [[1.3671827 ]
 [0.36286247]]
Epoch:  200  b_grad:  -2.8674110899458185  w_grad:  [[-0.12356135]
 [-0.08477796]]  b:  [80.45720355]  w:  [[1.46519403]
 [0.42126122]]
Epoch:  250  b_grad:  -1.0442241793037979  w_grad:  [[-0.05459422]
 [-0.04370804]]  b:  [81.368797]  w:  [[1.50765355]
 [0.45260181]]
Epoch:  300  b_grad:  -0.38027478531631814  w_grad:  [[-0.02471916]
 [-0.02166351]]  b:  [81.7007717]  w:  [[1.52662487]
 [0.46845214]]
Epoch:  350  b_grad:  -0.13848454691384535  w_grad:  [[

In [7]:
import plotly.graph_objects as go
fig3 = go.Figure()
fig3.add_trace(go.Scatter(y=losses,
                          mode='markers', name='loss',
                          marker=dict(color='black', size=7), opacity=0.8))
fig3.update_layout(title_text="MSE vs epoch", title_font_size=20,
                   xaxis_title="epoch", yaxis_title="MSE")
fig3.show()

In [8]:
import torch
from torch import nn
model = nn.Linear(bias=True, in_features=2, out_features=1)

In [9]:
x_torch = torch.as_tensor(x_s).float()
y_torch = torch.as_tensor(y).float()

In [10]:
from torch import optim
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [11]:
n_epochs = 500
losses = []

for epoch in range(n_epochs):
    loss = 0.0
    model.train()
    optimizer.zero_grad()
    y_hat_torch = model(x_torch)
    loss = loss_fn(y_hat_torch, y_torch)
    loss.backward()
    optimizer.step()
    losses.append(loss.detach().numpy())
    if epoch % 50 == 0:
        print('Epoch: {}, Loss: {:.2f}'.format(epoch, loss))

Epoch: 0, Loss: 6672.88
Epoch: 50, Loss: 885.17
Epoch: 100, Loss: 117.55
Epoch: 150, Loss: 15.73
Epoch: 200, Loss: 2.22
Epoch: 250, Loss: 0.43
Epoch: 300, Loss: 0.19
Epoch: 350, Loss: 0.16
Epoch: 400, Loss: 0.16
Epoch: 450, Loss: 0.16


In [12]:
print(model.bias, model.weight)

Parameter containing:
tensor([81.8876], requires_grad=True) Parameter containing:
tensor([[1.5423, 0.4830]], requires_grad=True)


In [13]:
fig3.add_trace(go.Scatter(y=losses,
                          mode='markers+lines', name='PyTorch loss',
                          marker=dict(color='red', size=3), opacity=0.8))
fig3.show()

In [14]:
import warnings

model.eval()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    _ = torch.onnx.export(
        model, x_torch[:1], "neuron_two_inputs.onnx",
        input_names=['x'], output_names=['y'])
print("Модель сохранена: neuron_two_inputs.onnx")

[torch.onnx] Obtain model graph for `Linear(in_features=2, out_features=1, bias=True)` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Linear(in_features=2, out_features=1, bias=True)` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Модель сохранена: neuron_two_inputs.onnx
